<a href="https://colab.research.google.com/github/m42tk7246v-png/JCU_MA3832_Stephenson/blob/main/d2l_chapter3_nn_classes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# D2L Chapter 3 — PyTorch `nn` Class Reference

All PyTorch classes used under the hood in Chapter 3: *Linear Neural Networks for Regression*.

| Class | Module | Role |
|---|---|---|
| `nn.Module` | `torch.nn` | Base class for all models |
| `nn.Linear` | `torch.nn` | Affine transformation $y = xW^\top + b$ |
| `nn.LazyLinear` | `torch.nn` | Like `Linear` but infers input size on first call |
| `nn.MSELoss` | `torch.nn` | Mean squared error loss |
| `torch.optim.SGD` | `torch.optim` | Stochastic gradient descent optimizer |
| `TensorDataset` | `torch.utils.data` | Zips tensors into an indexable dataset |
| `DataLoader` | `torch.utils.data` | Batches, shuffles, and iterates a dataset |

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

---
## `nn.Module`

The base class for every PyTorch model. Subclass it and implement `forward()`.

Key responsibilities:

- **Parameter tracking** — any `nn.Parameter` or `nn.Module` assigned as an attribute is
  automatically registered and returned by `.parameters()`
- **Mode switching** — `.train()` / `.eval()` toggle layers like dropout and batch norm
- **Device transfer** — `.to(device)` moves all parameters and buffers to CPU or GPU
- **Serialisation** — `.state_dict()` / `.load_state_dict()` for saving and loading weights

In [3]:
class MyModel(nn.Module):
    def __init__(self):
        # Python requirement for inherited classes: must call the parent __init__ to
        # set up nn.Module's internal machinery (parameter registry, hooks, etc.)
        super().__init__()
        # assigning an nn.Module as an attribute automatically registers it as a submodule;
        # its parameters will appear in model.parameters() and model.named_parameters()
        self.net = nn.LazyLinear(1)

    # nn.Module routes model(X) calls to this method via __call__
    def forward(self, X):
        return self.net(X)

model = MyModel()

# named_modules() returns all registered submodules (including self)
print('Submodules:', list(model.named_modules()))
# .training is True by default; toggled by .train() and .eval()
print('Training mode:', model.training)

Submodules: [('', MyModel(
  (net): LazyLinear(in_features=0, out_features=1, bias=True)
)), ('net', LazyLinear(in_features=0, out_features=1, bias=True))]
Training mode: True


In [4]:
# LazyLinear has no parameters until it sees data; the first forward pass
# infers in_features from X and materialises the weight and bias tensors
X_demo = torch.randn(5, 3)
_ = model(X_demo)

# named_parameters() yields (name, tensor) for every trainable parameter in the model tree
for name, param in model.named_parameters():
    print(f'{name:20s}  shape: {tuple(param.shape)}')

net.weight            shape: (1, 3)
net.bias              shape: (1,)


---
## `nn.Linear`

Applies the affine map $y = xW^\top + b$.

```python
nn.Linear(in_features, out_features, bias=True)
```

| Attribute | Shape | Description |
|---|---|---|
| `weight` | `(out_features, in_features)` | $W$, initialised uniformly on (-k, k) where k = 1 / sqrt(in_features) |
| `bias` | `(out_features,)` | $b$, same initialisation |

The from-scratch equivalent is simply:
```python
y = X @ w + b
```

In [ ]:
layer = nn.Linear(in_features=2, out_features=1)
# weight is stored as (out, in) — transposed relative to the mathematical convention Xw
print('weight:', layer.weight)   # shape (1, 2)
print('bias:  ', layer.bias)     # shape (1,)

X = torch.tensor([[1.0, 2.0],
                   [3.0, 4.0]])
print('output:', layer(X))       # shape (2, 1) — one scalar prediction per row

In [ ]:
# .data gives direct access to the underlying tensor, bypassing autograd —
# needed here because we want to set values without recording the operation
# trailing _ on normal_() and fill_() means the operation is in-place
layer.weight.data.normal_(0, 0.01)
layer.bias.data.fill_(0)  # in-place fill — see above
print('after init — weight:', layer.weight.data)
print('after init — bias:  ', layer.bias.data)

---
## `nn.LazyLinear`

Identical to `nn.Linear` except `in_features` is inferred automatically on the
first forward pass. This removes the need to know the input dimension at construction time.

```python
nn.LazyLinear(out_features, bias=True)
```

Before the first forward pass the layer is in a *lazy* state — `weight` and `bias`
are `UninitializedParameter` objects. After the first call they materialise as normal tensors.

This is why `LinearRegression.__init__` initialises weights **after** constructing
the layer but the weights only exist after the first forward pass.

In [ ]:
lazy = nn.LazyLinear(1)
# has_uninitialized_params() returns True while weight/bias are still placeholder objects
print('before forward — uninitialized:', lazy.has_uninitialized_params())

X = torch.randn(4, 3)   # 3 input features
out = lazy(X)
# in_features is now inferred as 3 from the shape of X
print('after forward  — weight shape:', lazy.weight.shape)  # (1, 3)
print('output shape:', out.shape)

---
## `nn.MSELoss`

Mean (or sum) squared error between predictions and targets.

```python
nn.MSELoss(reduction='mean')  # 'mean' | 'sum' | 'none'
```

With `reduction='mean'` (default):
$$\ell(\hat{y}, y) = \frac{1}{n} \sum_{i=1}^n (\hat{y}_i - y_i)^2$$

Note: this differs from `LinearRegressionScratch` by a factor of $\frac{1}{2}$,
which D2L includes to make the derivative cleaner:
$$\frac{d}{d\hat{y}} \ \left(\frac{1}{2}(\hat{y}-y)^2 \right) = (\hat{y}-y)$$

In [ ]:
loss_fn = nn.MSELoss()

y_hat = torch.tensor([2.5, 0.0, 2.0, 8.0])
y     = torch.tensor([3.0, -0.5, 2.0, 7.0])

# .item() extracts a plain Python scalar from a 0-dimensional tensor
print('nn.MSELoss:         ', loss_fn(y_hat, y).item())
print('by hand:            ', ((y_hat - y) ** 2).mean().item())        # identical
print('scratch (with 1/2): ', (0.5 * (y_hat - y) ** 2).mean().item()) # LinearRegressionScratch version

---
## `torch.optim.SGD`

Stochastic gradient descent (with optional momentum, weight decay, etc.).

```python
torch.optim.SGD(params, lr, momentum=0, weight_decay=0)
```

The update rule for plain SGD (`momentum=0`) is:
$$\theta \leftarrow \theta - \eta \nabla_\theta \ell$$

| Method | What it does |
|---|---|
| `.zero_grad()` | Clears accumulated gradients — call before each backward pass |
| `.step()` | Applies the update rule to all parameters |

`params` is typically `model.parameters()` — the iterator returned by `nn.Module`.

In [ ]:
model = nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

X = torch.randn(32, 2)
y = torch.randn(32, 1)

# PyTorch accumulates gradients by default, so clear them before each new backward pass
optimizer.zero_grad()
y_hat = model(X)            # forward pass: compute predictions
loss = loss_fn(y_hat, y)    # compute scalar loss
# backpropagation: traverses the computation graph and computes ∂loss/∂param
# for every parameter that has requires_grad=True
loss.backward()
# apply θ ← θ − η∇θ for each parameter registered in the optimizer
optimizer.step()

print(f'loss: {loss.item():.4f}')

In [ ]:
# Scratch equivalent — the same five steps with raw tensors instead of nn classes
# requires_grad=True tells PyTorch to track operations on these tensors
# so .backward() can compute their gradients
w = torch.zeros(2, 1, requires_grad=True)
b = torch.zeros(1,    requires_grad=True)
lr = 0.01

y_hat = X @ w + b
loss = (0.5 * (y_hat - y) ** 2).mean()
loss.backward()  # compute ∂loss/∂w and ∂loss/∂b

# torch.no_grad() disables gradient tracking for the block below —
# the parameter update must not itself be recorded in the computation graph
with torch.no_grad():
    w -= lr * w.grad          # θ ← θ − η∇θ  (same as optimizer.step())
    b -= lr * b.grad
    w.grad.zero_()            # in-place zero  (same as optimizer.zero_grad())
    b.grad.zero_()

print(f'scratch loss: {loss.item():.4f}')

---
## `TensorDataset` and `DataLoader`

These two classes together replace the manual shuffle-and-slice loop from the
NumPy minibatch implementation.

### `TensorDataset`

```python
TensorDataset(*tensors)
```

Zips a collection of tensors along their first dimension. Supports indexing so
a `DataLoader` can sample from it. All tensors must have the same `size(0)`.

### `DataLoader`

```python
DataLoader(dataset, batch_size, shuffle=True, num_workers=0)
```

Wraps a `Dataset` and returns an iterator that yields batches. With `shuffle=True`
it re-randomises the order each epoch — the same role as `np.random.permutation`
in the scratch implementation.

| Argument | Effect |
|---|---|
| `shuffle=True` | Reshuffles every epoch (use for training, not validation) |
| `num_workers` | Subprocesses for parallel data loading |
| `drop_last` | Drops the final batch if smaller than `batch_size` |

In [ ]:
X_data = torch.arange(20, dtype=torch.float).reshape(10, 2)
y_data = torch.arange(10, dtype=torch.float).reshape(10, 1)

# TensorDataset zips tensors along dim 0 so they can be indexed as matched pairs
dataset = TensorDataset(X_data, y_data)
# len() works because TensorDataset implements __len__
print(f'dataset length: {len(dataset)}')
# indexing works because TensorDataset implements __getitem__, returning a tuple
print(f'first item: {dataset[0]}')   # returns (X_data[0], y_data[0])

In [ ]:
loader = DataLoader(dataset, batch_size=3, shuffle=True)

# each iteration yields one (X_batch, y_batch) tuple; order is reshuffled each epoch
for X_batch, y_batch in loader:
    print(f'X_batch shape: {X_batch.shape}  y_batch shape: {y_batch.shape}')

In [ ]:
# DataModule.get_tensorloader() is simply:
#
#   tensors = tuple(a[indices] for a in tensors)   # slice train or val rows
#   dataset = TensorDataset(*tensors)
#   return DataLoader(dataset, self.batch_size, shuffle=train)
#
# The NumPy scratch equivalent was:
#
#   idx = np.random.permutation(n)                           # <-- shuffle=True
#   X_shuf, y_shuf = X[idx], y[idx]
#   for t in range(n_batches):
#       X_B = X_shuf[t * batch_size:(t + 1) * batch_size]   # <-- batch indexing
#       y_B = y_shuf[t * batch_size:(t + 1) * batch_size]

print('DataLoader replaces the manual shuffle-and-slice loop.')